# Customer Churn Intelligence — Leakage Audit

## Objective

Audit every column in the cleaned Telco Customer Churn dataset for **temporal / post-outcome leakage** and **process leakage** risks before train/validation/test splitting and modeling.

**Stage:** Step 7 — Leakage Audit (no split, no modeling, no SMOTE, no preprocessing pipelines).

## 1. Prediction-Time Definition

For this project, a churn prediction is made for an **active telecom customer at a snapshot in time**.

**Information allowed at prediction time (features):**
- Current account attributes known while the customer is still active (demographics, plan, services, contract type, billing preferences, tenure to date, current monthly charge, total charges accumulated so far).

**Information NOT allowed (post-outcome / post-churn leakage):**
- Any field created or finalized **after** the customer has churned (e.g., cancellation date, termination reason, final invoice, post-churn account status).
- The target itself (`Churn`) must never enter the feature matrix.

**Separate from temporal leakage:**
- **High correlation with churn is not leakage.** Predictive features may strongly associate with churn; we do not remove them for that reason alone.
- **`customerID`** is excluded because it is an identifier (memorization / non-generalizable), not because it is post-outcome data.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import IDENTIFIER_COL, TARGET_COL
from src.data_separation import (
    CATEGORICAL_FEATURE_COLS,
    FEATURE_COLS,
    NUMERIC_FEATURE_COLS,
    load_cleaned_data,
    separate_features_target_id,
)

# Forbidden post-outcome column name patterns (AGENTS.md rule 6)
FORBIDDEN_POST_OUTCOME_PATTERNS = [
    "cancel",
    "termination",
    "final invoice",
    "post-churn",
    "exit",
    "departure",
    "closed date",
    "end date",
]

df = load_cleaned_data()
separated = separate_features_target_id(df)

print(f"Dataset shape: {df.shape}")
print(f"Columns audited: {len(df.columns)}")

## 2. Scan for Forbidden Post-Outcome Columns

In [ ]:
def find_forbidden_columns(columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        col_lower = col.lower()
        matches = [p for p in FORBIDDEN_POST_OUTCOME_PATTERNS if p in col_lower]
        rows.append({"Column": col, "Forbidden Pattern Match": ", ".join(matches) if matches else None})
    result = pd.DataFrame(rows)
    return result[result["Forbidden Pattern Match"].notna()]


forbidden_hits = find_forbidden_columns(df.columns.tolist())
print(f"Forbidden post-outcome column name matches: {len(forbidden_hits)}")
display(forbidden_hits if len(forbidden_hits) else pd.DataFrame({"Result": ["None found"]}))

## 3. Column-by-Column Leakage Audit

In [ ]:
COLUMN_AUDIT: list[dict] = [
    {
        "Column": "customerID",
        "Role": "Identifier",
        "Available at Prediction Time": "Yes (as linkage key)",
        "Post-Outcome Leakage Risk": "None (not outcome-derived)",
        "Modeling Verdict": "Exclude from features",
        "Rationale": "Unique customer key for joining predictions; using it as a feature would cause memorization and poor generalization (AGENTS.md rule 5).",
    },
    {
        "Column": "gender",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Static demographic attribute present in customer record at snapshot time.",
    },
    {
        "Column": "SeniorCitizen",
        "Role": "Feature (numeric/binary)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Demographic flag (0/1) known at account creation / snapshot.",
    },
    {
        "Column": "Partner",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Household attribute recorded on active account.",
    },
    {
        "Column": "Dependents",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Household attribute recorded on active account.",
    },
    {
        "Column": "tenure",
        "Role": "Feature (numeric)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Months as customer up to snapshot; strongly predictive but not post-outcome.",
    },
    {
        "Column": "PhoneService",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current service subscription status.",
    },
    {
        "Column": "MultipleLines",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current phone add-on status; sentinel 'No phone service' reflects product mix, not missing outcome data.",
    },
    {
        "Column": "InternetService",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current internet product tier at snapshot.",
    },
    {
        "Column": "OnlineSecurity",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription; 'No internet service' is a product-absence sentinel.",
    },
    {
        "Column": "OnlineBackup",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription; sentinel category is not leakage.",
    },
    {
        "Column": "DeviceProtection",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription; sentinel category is not leakage.",
    },
    {
        "Column": "TechSupport",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription; high churn association in EDA is not grounds for removal.",
    },
    {
        "Column": "StreamingTV",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription.",
    },
    {
        "Column": "StreamingMovies",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current add-on subscription.",
    },
    {
        "Column": "Contract",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current contract type; strongly associated with churn but available before outcome.",
    },
    {
        "Column": "PaperlessBilling",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current billing preference on active account.",
    },
    {
        "Column": "PaymentMethod",
        "Role": "Feature (categorical)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current payment method; not a post-cancellation record.",
    },
    {
        "Column": "MonthlyCharges",
        "Role": "Feature (numeric)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Current recurring charge at snapshot time.",
    },
    {
        "Column": "TotalCharges",
        "Role": "Feature (numeric)",
        "Available at Prediction Time": "Yes",
        "Post-Outcome Leakage Risk": "None observed",
        "Modeling Verdict": "Approved",
        "Rationale": "Cumulative charges up to snapshot (cleaned: blank values for tenure=0 set to 0.0 via deterministic rule, not target-derived). Correlated with tenure but not post-outcome.",
    },
    {
        "Column": "Churn",
        "Role": "Target",
        "Available at Prediction Time": "No (outcome label)",
        "Post-Outcome Leakage Risk": "Target itself — must never be a feature",
        "Modeling Verdict": "Target only",
        "Rationale": "Binary outcome to predict; excluded from X by src/data_separation.py.",
    },
]

audit_df = pd.DataFrame(COLUMN_AUDIT)

# Verify audit covers every column exactly once
audited_cols = set(audit_df["Column"])
dataset_cols = set(df.columns)
assert audited_cols == dataset_cols, f"Audit mismatch: missing={dataset_cols - audited_cols}, extra={audited_cols - dataset_cols}"

audit_df

## 4. Audit Summary Tables

In [ ]:
approved_features = audit_df.loc[audit_df["Modeling Verdict"] == "Approved", "Column"].tolist()
identifier_cols = audit_df.loc[audit_df["Role"] == "Identifier", "Column"].tolist()
target_cols = audit_df.loc[audit_df["Role"] == "Target", "Column"].tolist()
post_outcome_leakage = audit_df.loc[
    audit_df["Post-Outcome Leakage Risk"].str.contains("Target itself|outcome-derived", case=False, na=False)
    | audit_df["Modeling Verdict"].eq("Exclude from features")
]

summary = pd.DataFrame(
    {
        "Category": [
            "Approved predictive features",
            "Identifier-only columns",
            "Target column",
            "Post-outcome leakage columns found",
            "Forbidden column name matches",
        ],
        "Count": [
            len(approved_features),
            len(identifier_cols),
            len(target_cols),
            0,
            len(forbidden_hits),
        ],
        "Columns": [
            ", ".join(approved_features),
            ", ".join(identifier_cols),
            ", ".join(target_cols),
            "None",
            "None" if len(forbidden_hits) == 0 else ", ".join(forbidden_hits["Column"]),
        ],
    }
)
summary

In [ ]:
# Confirm approved features match src/data_separation.py FEATURE_COLS
assert set(approved_features) == set(FEATURE_COLS)
assert identifier_cols == [IDENTIFIER_COL]
assert target_cols == [TARGET_COL]
assert IDENTIFIER_COL not in separated.X.columns
assert TARGET_COL not in separated.X.columns

print("Approved features match FEATURE_COLS in src/data_separation.py")
print(f"Feature matrix shape (X): {separated.X.shape}")

## 5. Process & Preprocessing Leakage Protections (Future Steps)

No temporal post-outcome columns were found in this dataset. Future **process leakage** must still be prevented during modeling:

| Risk | Planned protection |
|------|-------------------|
| Target leakage into features | Keep `Churn` out of `X`; enforce via `separate_features_target_id()` |
| Identifier memorization | Exclude `customerID` from all sklearn pipelines |
| Fit-on-full-data preprocessing | Fit `ColumnTransformer` / scalers / encoders on **training fold only** inside `Pipeline` |
| Test-set peeking | Hold out final test set until model + threshold policy are frozen (AGENTS.md rule 3) |
| SMOTE leakage | Apply resampling **after** split and **inside** cross-validation on training data only |
| Threshold / calibration leakage | Tune decision threshold and calibrate using validation split only |
| Duplicate preprocessing logic | Single saved pipeline shared by training, Streamlit, and FastAPI |
| TotalCharges cleaning | Already handled with deterministic tenure=0 → 0.0 rule (not target-based) |

## Leakage Audit Conclusion

**Temporal / post-outcome leakage found:** **None** — the dataset contains no cancellation dates, termination reasons, final invoices, or other post-churn fields.

**Excluded from modeling (non-temporal reasons):**
- `customerID` — identifier only (linkage key)

**Target:**
- `Churn` — outcome label only

**Approved predictive features (19):** all columns in `FEATURE_COLS` from `src/data_separation.py`.

Strong churn associations observed in focused EDA (e.g., `Contract`, `tenure`, `InternetService`) are **retained** — correlation alone is not leakage.

**Next step (not performed here):** Train/validation/test split with leakage-safe preprocessing pipelines.